# 03 · Unity Catalog: almacenamiento, auditoría y linaje

⏱ **10 min** · Módulo 4 · Slides de la Intro: 17 (auditoría y linaje) y 19 (acceso seguro al almacenamiento)

**Objetivo:** saber dónde se guardan físicamente los datos y responder dos preguntas de gobierno: quién hizo qué y de dónde viene cada dato.

| Paso | Qué haces |
|---|---|
| 1 | ¿Dónde están guardados mis datos? |
| 2 | Storage credential y external location |
| 3 | Auditoría: ¿quién hizo qué? |
| 4 | Linaje: ¿de dónde viene el dato? |
| 5 | Azure vs Free Edition |

**Requiere:** notebook 01.

In [0]:
%run ./00_Setup

## Paso 1 · ¿Dónde están guardados mis datos?

In [0]:
display(spark.sql("DESCRIBE TABLE EXTENDED suscriptores")
        .where("col_name IN ('Catalog', 'Database', 'Table', 'Type', 'Owner', 'Location')"))

In [0]:
%sql
DESCRIBE VOLUME datos

👀 **Qué observar:** la tabla y el volumen son **administrados** (`MANAGED`): Databricks decide dónde se guardan los archivos y los
borra cuando borras el objeto. En la Free Edition la ubicación sale vacía porque el almacenamiento es de Databricks; en Azure verías
una ruta `abfss://contenedor@cuenta.dfs.core.windows.net/...` dentro de **tu** ADLS Gen2.

## Paso 2 · Storage credential y external location

Son los dos objetos del slide 19: con ellos Unity Catalog entra a tu almacenamiento sin repartir llaves.

In [0]:
%sql
SHOW STORAGE CREDENTIALS

In [0]:
%sql
SHOW EXTERNAL LOCATIONS

👀 **Qué observar:** las dos listas salen vacías: la Free Edition no tiene almacenamiento propio que registrar.
¿Qué pasa si intentas guardar una tabla en una ruta que nadie registró?

In [0]:
try:
    spark.sql("CREATE TABLE pedidos_externa (id INT) LOCATION 's3://un-bucket-sin-registrar/pedidos'")
except Exception as e:
    print(str(e).splitlines()[0])

👀 **Qué observar:** Unity Catalog lo rechaza: no hay una *external location* que cubra esa ruta. Nadie escribe en el almacenamiento
por fuera del gobierno. En Azure, los pasos del laboratorio del slide 19 serían:

```sql
-- 1 y 2. Activar Unity Catalog y crear el metastore: lo hace el administrador en la consola de la cuenta
-- 3. Crear un catálogo con su ubicación
CREATE CATALOG prod MANAGED LOCATION 'abfss://datos@micuenta.dfs.core.windows.net/prod';

-- 4. Registrar ubicaciones
--    Storage credential: la identidad autorizada (un access connector de Azure).
--    Se crea en Catalog → External data → Credentials.
--    External location: una ruta de ADLS Gen2 + la credencial que la protege.
CREATE EXTERNAL LOCATION lago_ventas
  URL 'abfss://datos@micuenta.dfs.core.windows.net/ventas'
  WITH (STORAGE CREDENTIAL credencial_lago);

-- Una tabla externa: tú eliges la ruta; si borras la tabla, los archivos se quedan
CREATE TABLE prod.retail.pedidos_ext (pedido_id INT, monto DOUBLE)
  LOCATION 'abfss://datos@micuenta.dfs.core.windows.net/ventas/pedidos';
```

## Paso 3 · Auditoría: ¿quién hizo qué?

Tres lugares donde queda registrado, del más cercano al más amplio.

In [0]:
%sql
-- 1. El historial de la tabla: quién la cambió y cuándo
SELECT version, timestamp, userName, operation
FROM (DESCRIBE HISTORY suscriptores)
ORDER BY version

In [0]:
# 2. El historial de consultas: quién consultó qué (tarda unos minutos en reflejar lo último)
try_sql("""
    SELECT start_time, executed_by, statement_type, left(statement_text, 70) AS consulta
    FROM system.query.history
    WHERE statement_text ILIKE '%suscriptores%'
    ORDER BY start_time DESC
    LIMIT 10
""")

In [0]:
# 3. El registro de auditoría de la cuenta: quién creó, borró o dio permisos
try_sql("""
    SELECT event_time, user_identity.email AS usuario, action_name AS accion,
           request_params['schema_name'] AS esquema, request_params['name'] AS objeto
    FROM system.access.audit
    WHERE service_name = 'unityCatalog'
      AND action_name IN ('createSchema', 'createTable', 'createVolume', 'deleteTable', 'updatePermissions')
    ORDER BY event_time DESC
    LIMIT 10
""")

👀 **Qué observar:** el catálogo `system` son las **tablas del sistema**: auditoría, linaje, consumo. Se llenan solas.
Llegan **con retraso** (hasta unos 20 minutos): si todavía no ves lo que acabas de hacer, vuelve a ejecutar la celda más tarde.

## Paso 4 · Linaje: ¿de dónde viene el dato?

Creamos una tabla a partir de otra. Unity Catalog anota la relación sin que configures nada.

In [0]:
%sql
CREATE OR REPLACE TABLE suscriptores_por_plan AS
SELECT plan, count(*) AS suscriptores FROM suscriptores GROUP BY plan

👀 **Míralo en la interfaz:** **Catalog → lab_tvmaze_m56 → suscriptores_por_plan → pestaña Lineage → See lineage graph**.
Verás `suscriptores → suscriptores_por_plan` y el notebook que la creó.

El mismo linaje se puede consultar con SQL (aparece hasta unos 20 minutos después; mientras tanto verás el de los laboratorios anteriores):

In [0]:
try_sql("""
    SELECT DISTINCT source_table_full_name AS origen, target_table_full_name AS destino, entity_type AS creado_por
    FROM system.access.table_lineage
    WHERE target_table_schema IN ('lab_tvmaze', 'lab_tvmaze_m34', 'lab_tvmaze_m56')
      AND source_table_full_name IS NOT NULL
    ORDER BY destino
""")

## Paso 5 · Azure vs Free Edition

| Slide 19 (Azure) | Free Edition | En este laboratorio |
|---|---|---|
| 1. Activar Unity Catalog | Ya viene activado | — |
| 2. Crear el metastore | Ya creado y asignado | `current_metastore()` (notebook 01) |
| 3. Crear un catálogo | Se puede crear | `CREATE CATALOG lab_prod` (notebook 01) |
| 4. Registrar ubicaciones | No hay almacenamiento propio | Listas vacías y el error del paso 2 |
| Grupos (`analistas`, `finanzas`) | Grupo `account users` | `GRANT ... TO account users` (notebook 02) |

## ✔ Checkpoint

- [ ] Sé qué es una tabla administrada y qué cambiaría con una tabla externa
- [ ] Sé qué son una *storage credential* y una *external location*
- [ ] Encontré quién cambió una tabla y quién la consultó
- [ ] Vi el linaje de una tabla en Catalog Explorer

**Siguiente:** `04_ETL_Medallion` (empieza el módulo 5)